# Phase 2: Bitcoin Price Data - Interactive Story & Dashboard Preparation

## Setup & Data Loading

In [39]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import os

path = "bitcoin-prices-dataset"
csv_file = [f for f in os.listdir(path) if f.endswith('.csv')][0]
df = pd.read_csv(os.path.join(path, csv_file))

df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)

print(f"Dataset Overview:")
print(f"  Date Range: {df['Date'].min().date()} to {df['Date'].max().date()}")
print(f"  Total Records: {len(df)}")
print(f"  Sample:\n {df.head()}")


print(f"\nSome Statistics:\n")
print(df[['Open', 'High', 'Low', 'Close', 'Volume']].describe())

Dataset Overview:
  Date Range: 2010-01-01 to 2026-02-08
  Total Records: 5883
  Sample:
         Date  Open      High       Low  Close  Volume PriceCategory
0 2010-01-01   0.3  0.303428  0.295510    0.3   715.8           Low
1 2010-01-02   0.3  0.304377  0.299459    0.3  2028.3           Low
2 2010-01-03   0.3  0.303736  0.295858    0.3   273.0           Low
3 2010-01-04   0.3  0.303406  0.298489    0.3  1452.0           Low
4 2010-01-05   0.3  0.302715  0.296330    0.3  1002.6           Low

Some Statistics:

               Open          High           Low         Close        Volume
count   5883.000000   5883.000000   5883.000000   5883.000000  5.883000e+03
mean   21725.051095  21973.716194  21487.576193  21736.779754  1.087915e+08
std    30114.978682  30445.256528  29793.029792  30119.952256  1.851098e+08
min        0.300000      0.300327      0.294003      0.300000  3.030000e+01
25%      430.570000    431.211444    422.134483    430.570000  1.271205e+06
50%     7193.600000   7207.

## Key Feature Engineering

In [40]:
# Calculate key metrics that will drive our narrative

# Measures the percentage range of price movement within a day
df['Daily_Volatility'] = ((df['High'] - df['Low']) / df['Low']) * 100

# Direction and magnitude of daily movement
df['Price_Change'] = ((df['Close'] - df['Open']) / df['Open']) * 100

# 30-day trends
df['MA_30'] = df['Close'].rolling(window=30).mean()

# 30-day rolling average of volatility
df['Volatility_MA_30'] = df['Daily_Volatility'].rolling(window=30).mean()

# Year and Month for grouping analysis
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['YearMonth'] = df['Date'].dt.to_period('M')

print(f"\nFeature Summary:")
print(df[['Daily_Volatility', 'Price_Change', 'Volatility_MA_30']].describe())


Feature Summary:
       Daily_Volatility  Price_Change  Volatility_MA_30
count       5883.000000   5883.000000       5854.000000
mean           3.833889      1.553441          3.842383
std           77.290527     76.124849         14.059741
min            0.209324    -73.561781          1.616316
25%            1.522888      0.000000          2.022211
50%            2.092616      0.000000          2.121155
75%            2.696399      0.000000          2.237626
max         5544.002504   5472.875092        187.106743


## Extract Main Insights

In [46]:
#  Price evolution and market phases
min_price = df['Close'].min()
max_price = df['Close'].max()
current_price = df['Close'].iloc[-1]
total_return = ((current_price - df['Close'].iloc[0]) / df['Close'].iloc[0]) * 100

insight_1 = f"""
INSIGHT 1: PRICE EVOLUTION & MARKET PHASES
• Starting Price (2010): ${df['Close'].iloc[0]:.4f}
• Current Price: ${current_price:.2f}
• Total Return: {total_return:.1f}%
• Price Range: ${min_price:.2f} to ${max_price:.2f}
• Market Phases: {df['PriceCategory'].value_counts().to_dict()}
"""

# Volatility patterns
high_volatility_days = (df['Daily_Volatility'] > df['Daily_Volatility'].quantile(0.75)).sum()
avg_volatility = df['Daily_Volatility'].mean()

insight_2 = f"""
INSIGHT 2: VOLATILITY CHARACTERISTICS
• Average Daily Volatility: {avg_volatility:.2f}%
• High Volatility Days (top 25%): {high_volatility_days} days
• Peak Volatility: {df['Daily_Volatility'].max():.2f}% on {df.loc[df['Daily_Volatility'].idxmax(), 'Date'].date()}
• Most Volatile Period: {df.loc[df['Volatility_MA_30'].idxmax(), 'Date'].strftime('%b %Y')}
"""


# Price category distribution
category_stats = df.groupby('PriceCategory').agg({
    'Close': ['min', 'max', 'mean'],
    'Date': 'count'
}).round(2)

insight_3 = f"""
INSIGHT 3: MARKET PHASE ANALYSIS
{category_stats.to_string()}
"""

print(insight_1)
print(insight_2)
print(insight_3)


INSIGHT 1: PRICE EVOLUTION & MARKET PHASES
• Starting Price (2010): $0.3000
• Current Price: $69000.00
• Total Return: 22999900.0%
• Price Range: $0.30 to $93429.20
• Market Phases: {'Low': 2191, 'High': 1866, 'Medium': 1826}


INSIGHT 2: VOLATILITY CHARACTERISTICS
• Average Daily Volatility: 3.83%
• High Volatility Days (top 25%): 1471 days
• Peak Volatility: 5544.00% on 2013-01-01
• Most Volatile Period: Jan 2013


INSIGHT 3: MARKET PHASE ANALYSIS
                  Close                      Date
                    min       max      mean count
PriceCategory                                    
High           29001.72  93429.20  59898.23  1866
Low                0.30    754.01    253.76  2191
Medium           963.74  16547.50   8516.65  1826



## Interactive Visualizations

### The Rise of Bitcoin - Price Evolution with Market Phases

In [ ]:
# Interactive plot showing Bitcoin's price
fig = go.Figure()

category_colors = {
    'Low':       '#1f77b4',      # Blue
    'Medium':    '#ff7f0e',   # Orange
    'High':      '#d62728',     # Red
    'Very High': '#9467bd'  # Purple
}

# Plot price with category coloring
for category in df['PriceCategory'].unique():
    mask = df['PriceCategory'] == category
    fig.add_trace(go.Scatter(
        x=df[mask]['Date'],
        y=df[mask]['Close'],
        name=f'Price - {category}',
        mode='lines',
        line=dict(color=category_colors.get(category, '#000000'), width=2),
        hovertemplate='<b>%{x|%Y-%m-%d}</b><br>Price: $%{y:.2f}<extra></extra>'
    ))

# Add moving averages
fig.add_trace(go.Scatter(
    x=df['Date'],
    y=df['MA_30'],
    name='30-Day MA',
    mode='lines',
    line=dict(color='rgba(0,255,0,0.5)', width=2, dash='dash'),
    hovertemplate='30-Day MA: $%{y:.2f}<extra></extra>'
))

fig.update_layout(
    title=dict(
        text='<b>Bitcoin Price Journey</b><br><sub>Color represents market phase intensity</sub>',
        x=0.5,
        xanchor='center'
    ),
    xaxis_title='Date',
    yaxis_title='Price (USD)',
    hovermode='x unified',
    template='plotly_white',
    height=600,
    yaxis=dict(type='log'),  # Log scale to show early and recent periods equally
    font=dict(size=12)
)

fig.show()

print(" Bitcoin's exponential price growth shows distinct market phases")

 Bitcoin's exponential price growth shows distinct market phases


### Volume & Activity - The Pulse of the Market

In [43]:
# Volume analysis with price overlay
fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=('Bitcoin Price', 'Trading Volume'),
    shared_xaxes=True,
    specs=[[{'secondary_y': False}], [{'secondary_y': False}]]
)

# Price trace
fig.add_trace(
    go.Scatter(
        x=df['Date'],
        y=df['Close'],
        name='Price',
        mode='lines',
        line=dict(color='#1f77b4', width=2),
        hovertemplate='Price: $%{y:.2f}<extra></extra>'
    ),
    row=1, col=1
)

# Volume bars with color based on price movement
colors = ['#d62728' if df.iloc[i]['Close'] < df.iloc[i]['Open'] else '#2ca02c' 
          for i in range(len(df))]

fig.add_trace(
    go.Bar(
        x=df['Date'],
        y=df['Volume'],
        name='Volume',
        marker=dict(color=colors, opacity=0.6),
        hovertemplate='Volume: %{y:,.0f} BTC<extra></extra>'
    ),
    row=2, col=1
)

fig.update_yaxes(title_text='Price (USD)', row=1, col=1, type='log')
fig.update_yaxes(title_text='Volume (BTC)', row=2, col=1)
fig.update_xaxes(title_text='Date', row=2, col=1)

fig.update_layout(
    title=dict(
        text='<b>Price & Volume Relationship: Market Momentum</b><br><sub>Green = price up, Red = price down</sub>',
        x=0.5,
        xanchor='center'
    ),
    height=700,
    template='plotly_white',
    hovermode='x unified'
)

fig.show()


### Market Phases - The Evolution Stages

In [47]:
# Market phase distribution and characteristics
phase_data = df.groupby('PriceCategory').agg({
    'Close': ['count', 'min', 'max', 'mean'],
    'Daily_Volatility': 'mean',
    'Volume': 'mean'
}).round(2)

# Create a comprehensive phase comparison
phase_summary = []
for category in df['PriceCategory'].unique():
    mask = df['PriceCategory'] == category
    phase_summary.append({
        'Phase': category,
        'Days': mask.sum(),
        'Min Price': df[mask]['Close'].min(),
        'Max Price': df[mask]['Close'].max(),
        'Avg Price': df[mask]['Close'].mean(),
        'Avg Volatility': df[mask]['Daily_Volatility'].mean(),
        'Avg Volume': df[mask]['Volume'].mean()
    })

phase_df = pd.DataFrame(phase_summary).sort_values('Min Price')

# Create visualizations
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=('Days in Each Phase', 'Avg Volatility by Phase', 'Avg Volume by Phase')
)

# Plot 1: Days in phase
fig.add_trace(
    go.Bar(
        x=phase_df['Phase'],
        y=phase_df['Days'],
        name='Days',
        marker=dict(color='#1f77b4'),
        hovertemplate='%{x}<br>Days: %{y}<extra></extra>'
    ),
    row=1, col=1
)

# Plot 2: Avg volatility
fig.add_trace(
    go.Bar(
        x=phase_df['Phase'],
        y=phase_df['Avg Volatility'],
        name='Volatility',
        marker=dict(color='#d62728'),
        hovertemplate='%{x}<br>Avg Volatility: %{y:.2f}%<extra></extra>'
    ),
    row=1, col=2
)

# Plot 3: Avg volume
fig.add_trace(
    go.Bar(
        x=phase_df['Phase'],
        y=phase_df['Avg Volume'],
        name='Volume',
        marker=dict(color='#2ca02c'),
        hovertemplate='%{x}<br>Avg Volume: %{y:,.0f}<extra></extra>'
    ),
    row=1, col=3
)

fig.update_yaxes(title_text='Days', row=1, col=1)
fig.update_yaxes(title_text='Volatility (%)', row=1, col=2)
fig.update_yaxes(title_text='Volume (BTC)', row=1, col=3)

fig.update_layout(
    title=dict(
        text='<b>Bitcoin Market Phases: Characteristics & Duration</b>',
        x=0.5,
        xanchor='center'
    ),
    height=500,
    template='plotly_white',
    showlegend=False
)

fig.show()

## Dashboard Strategy

In [ ]:
summary = f"""
╔════════════════════════════════════════════════════════════════════════════════╗
║                    BITCOIN DATA STORY - KEY NARRATIVE POINTS                    ║
╚════════════════════════════════════════════════════════════════════════════════╝

THE STORY:
   Bitcoin price has evolved through distinct market phases, from early obscurity
   (Low) to recent mainstream adoption (Very High). Each phase shows unique risk
   and volatility characteristics.

KEY METRICS FOR DASHBOARD:
   1. Price Trend (Log Scale) with Phase Coloring
      → Shows long-term trajectory and market evolution
   
   2. Volatility Indicator (30-Day MA)
      → Identifies risk periods and investment windows
   
   3. Volume & Momentum (Price/Volume Correlation)
      → Shows market participation and sentiment
   
   4. Phase Comparison (Risk/Duration/Activity)
      → Highlights phase-specific characteristics
   
   5. Risk Profile (Volatility Distribution by Phase)
      → Informs trading strategy decisions

INSIGHTS FOR DASHBOARD FOCUS:
   • High-priced phases show {phase_df.loc[phase_df['Avg Volatility'].idxmax(), 'Phase']} 
     with avg volatility of {phase_df['Avg Volatility'].max():.2f}%
   
   • Volume patterns suggest increased activity in higher price phases
   
   • Date range spans {df['Date'].min().strftime('%Y')} to {df['Date'].max().strftime('%Y')}
     = {(df['Date'].max() - df['Date'].min()).days} trading days

DASHBOARD WILL LOOK LIKE THIS:
   ┌─────────────────────────────────────┐
   │   Price Evolution (Main Chart)      │
   │   - Phase coloring                  │
   │   - Moving averages                 │
   └─────────────────────────────────────┘
   ┌──────────────────┬──────────────────┐
   │ Volatility Trend │ Phase Metrics    │
   │ (Risk Gauge)     │ (KPIs)           │
   └──────────────────┴──────────────────┘
   ┌──────────────────┬──────────────────┐
   │ Volume & Price   │ Risk Distribution│
   │ (Momentum)       │ (Box Plots)      │
   └──────────────────┴──────────────────┘

"""

print(summary)


╔════════════════════════════════════════════════════════════════════════════════╗
║                    BITCOIN DATA STORY - KEY NARRATIVE POINTS                    ║
╚════════════════════════════════════════════════════════════════════════════════╝

🎯 THE STORY:
   Bitcoin price has evolved through distinct market phases, from early obscurity
   (Low) to recent mainstream adoption (Very High). Each phase shows unique risk
   and volatility characteristics.

📊 KEY METRICS FOR DASHBOARD:
   1. Price Trend (Log Scale) with Phase Coloring
      → Shows long-term trajectory and market evolution

   2. Volatility Indicator (30-Day MA)
      → Identifies risk periods and investment windows

   3. Volume & Momentum (Price/Volume Correlation)
      → Shows market participation and sentiment

   4. Phase Comparison (Risk/Duration/Activity)
      → Highlights phase-specific characteristics

   5. Risk Profile (Volatility Distribution by Phase)
      → Informs trading strategy decisions

💡 INSIG